In [8]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi

no_preview = False # Toggle to False to use the preview widget


%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

장면 로딩 및 객체 병합

In [9]:
from sionna.rt.scene import load_scene

scene = load_scene("/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml")

#cam = Camera(position=[0, 600, -2000], look_at=[0, 0, 0])
cam = Camera(position=[0, 200, -1000], look_at=[0, 0, 0])
if no_preview:
    scene.render(camera=cam);
else:
    scene.preview();


In [10]:
for name, obj in scene.objects.items():
    print(f'{name:<15}{obj.radio_material.name}')

elm__23        wall
elm__24        roof
elm__25        8b4513
elm__26        2f4f4f
elm__27        red
elm__28        white
elm__29        gray
elm__30        black
elm__31        darkgrey
elm__32        grey
elm__33        lightgrey
elm__34        silver
elm__35        brown
elm__36        d2aa6d
elm__37        a58e9a
elm__38        ffe0a0
elm__39        dda088
elm__40        f4a460
elm__41        green
elm__42        696969
elm__43        f4e1c3
elm__44        5a564b
elm__54        forest
elm__50        water
elm__52        areas_pedestrian
elm__48        vegetation
elm__89        itu_concrete
elm__00        roads_primary
elm__46        areas_railways


도로 색 변경,좌표 추출

In [11]:
import mitsuba as mi
import drjit as dr
from sionna.rt import load_scene, Camera
from sionna.rt import ITURadioMaterial

# 1. 장면 로드
scene = load_scene("/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml")

target_id = "elm__00"

if target_id in scene.objects:
    # [수정] thickness 매개변수 추가 (예: 0.2 미터)
    red_road_mat = ITURadioMaterial(name="red_road_mat",
                                    itu_type="concrete",
                                    thickness=2,  # <--- 이 부분을 추가하세요
                                    color=[1.0, 0.0, 0.0]) 
    
    # 3. 씬에 재질 등록
    scene.add(red_road_mat)
    
    # 4. 도로 객체에 새 재질 할당
    scene.objects[target_id].radio_material = "red_road_mat"
    
    print(f"[성공] '{target_id}'에 빨간색 재질(두께 0.2m)을 적용했습니다.")
else:
    print(f"[실패] '{target_id}' 객체를 찾을 수 없습니다.")

# 5. 시각화
cam = Camera(position=[0, 200, 1000], look_at=[0, 0, 0])

try:
    if 'no_preview' in globals() and no_preview:
        print("Preview skipped.")
    else:
        scene.preview()
except:
    scene.preview()

[성공] 'elm__00'에 빨간색 재질(두께 0.2m)을 적용했습니다.


In [12]:
road_object_id = "elm__00"
road_positions = []

print(f"[탐색] Mitsuba Scene 내부에서 '{road_object_id}' 형상을 찾습니다...")

if hasattr(scene, 'mi_scene'):
    mi_scene = scene.mi_scene
    
    # 1. Mitsuba Scene의 모든 Shape를 순회하며 ID 매칭
    target_shape = None
    for s in mi_scene.shapes():
        if s.id() == road_object_id:
            target_shape = s
            break
    
    if target_shape: 
        try:           
            params = mi.traverse(target_shape)
                        
            if 'vertex_positions' in params:
                vertex_buffer = params['vertex_positions']
                
                # NumPy 변환 (1차원 배열: x, y, z, x, y, z ...)
                vertices_flat = np.array(vertex_buffer)
                
                # (N, 3) 형태로 변환 (x, y, z)
                if len(vertices_flat) > 0:
                    vertices = vertices_flat.reshape(-1, 3)
                    road_positions = vertices
                    print(f"  -> 추출된 도로 좌표 수: {len(road_positions)}개")
                else:
                    print("  [경고] 버퍼가 비어있습니다.")
            else:
                print(f"[오류] '{road_object_id}' 객체에 'vertex_positions' 속성이 없습니다.")

        except Exception as e:
            print(f"[오류] Vertex 추출 중 에러 발생: {e}")
    else:
        print(f"[실패] ID가 '{road_object_id}'인 Shape를 mi_scene에서 찾을 수 없습니다.")
else:
    print("[오류] scene 객체에서 'mi_scene' 속성을 찾을 수 없습니다.")

[탐색] Mitsuba Scene 내부에서 'elm__00' 형상을 찾습니다...
  -> 추출된 도로 좌표 수: 610개


In [13]:
import plotly.graph_objects as go
import numpy as np

# 1. 데이터 준비
x = road_positions[:, 0]
y = road_positions[:, 1]
indices = list(range(len(road_positions)))

# 2. 인터랙티브 지도 생성
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x, y=y,
    mode='markers',
    marker=dict(size=5, color='blue'),
    text=indices,  # 마우스를 올리면 이 번호(Index)가 뜸
    hovertemplate='<b>Index: %{text}</b><br>X: %{x:.1f}<br>Y: %{y:.1f}<extra></extra>'
))

fig.update_layout(
    title="도로 점 확인용 지도 (마우스를 올려보세요)",
    width=1000, height=800,
    hovermode='closest'
)

fig.show()

In [14]:
import numpy as np

# 데이터의 범위(Range) 확인
x_range = np.ptp(road_positions[:, 0]) # Peak to Peak (Max - Min)
y_range = np.ptp(road_positions[:, 1])
z_range = np.ptp(road_positions[:, 2])

print(f"X축 변화량: {x_range:.2f}")
print(f"Y축 변화량: {y_range:.2f}")
print(f"Z축 변화량: {z_range:.2f}")

# 높이 축 판단
if y_range < x_range and y_range < z_range:
    print("\n[분석 결과] Y축의 변화가 가장 작습니다. -> **Y축이 높이(Height)**일 확률이 높습니다.")
    print("           따라서 지도를 그리려면 **X축과 Z축**을 사용해야 합니다.")
elif z_range < x_range and z_range < y_range:
    print("\n[분석 결과] Z축의 변화가 가장 작습니다. -> **Z축이 높이(Height)**입니다.")
    print("           기존대로 X축과 Y축을 사용하는 것이 맞으나, 도로가 일직선일 수 있습니다.")

X축 변화량: 718.83
Y축 변화량: 0.00
Z축 변화량: 1044.20

[분석 결과] Y축의 변화가 가장 작습니다. -> **Y축이 높이(Height)**일 확률이 높습니다.
           따라서 지도를 그리려면 **X축과 Z축**을 사용해야 합니다.


In [15]:
import plotly.graph_objects as go

# [수정] Y축이 높이라면, 지도의 '세로'는 Z축 데이터가 되어야 함
x_vals = road_positions[:, 0]
y_vals = road_positions[:, 2] # 여기가 Z축 데이터로 변경됨 (index 1 -> 2)
indices = list(range(len(road_positions)))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_vals, y=y_vals,
    mode='markers',
    marker=dict(size=5, color='blue'),
    text=indices,
    hovertemplate='<b>Index: %{text}</b><br>X: %{x:.1f}<br>Z: %{y:.1f}<extra></extra>' # 라벨도 Z로 표기
))

fig.update_layout(
    title="도로 점 확인용 지도 (X - Z 평면)",
    xaxis_title="X Axis (East/West)",
    yaxis_title="Z Axis (North/South)", # Y축 라벨을 Z축으로 변경
    width=1000, height=800,
    hovermode='closest'
)

fig.show()

In [16]:
import numpy as np
import pandas as pd  # 데이터 보기 편하게 하기 위해 Pandas 사용 (선택 사항)

# 1. 데이터 검증 (Data Verification)
if 'road_positions' in globals() and len(road_positions) > 0:
    
    # --- [Step 1] 모든 좌표 출력 (Print All Coordinates) ---
    print(f"Start printing all {len(road_positions)} points...")
    print("-" * 50)
    print(f"{'Index':<6} | {'X (East/West)':<12} | {'Y (Height)':<12} | {'Z (North/South)':<12}")
    print("-" * 50)
    
    for i, pos in enumerate(road_positions):
        # 소수점 2자리까지만 출력하여 가독성 확보
        print(f"{i:<6} | {pos[0]:<12.2f} | {pos[1]:<12.2f} | {pos[2]:<12.2f}")
        
    print("-" * 50)
    print("[완료] 모든 좌표 출력이 끝났습니다.\n")

    # --- [Step 2] 축 데이터 범위 분석 (Axis Range Analysis) ---
    # 확실한 근거를 위해 각 축의 변화량을 계산합니다.
    x_min, x_max = np.min(road_positions[:, 0]), np.max(road_positions[:, 0])
    y_min, y_max = np.min(road_positions[:, 1]), np.max(road_positions[:, 1])
    z_min, z_max = np.min(road_positions[:, 2]), np.max(road_positions[:, 2])

    print("=" * 40)
    print("[데이터 분석 결과 / Data Analysis Result]")
    print(f"X축 범위: {x_min:.2f} ~ {x_max:.2f} (변화량: {x_max - x_min:.2f})")
    print(f"Y축 범위: {y_min:.2f} ~ {y_max:.2f} (변화량: {y_max - y_min:.2f})")
    print(f"Z축 범위: {z_min:.2f} ~ {z_max:.2f} (변화량: {z_max - z_min:.2f})")
    print("=" * 40)

    # --- [Step 3] 해석 (Interpretation) ---
    if (y_max - y_min) < 0.1: # 변화량이 거의 없다면
        print(">> 분석: Y축의 값이 거의 변하지 않습니다.")
        print(">> 결론: Y축이 '높이(Height)'를 나타내며, 도로가 평지임을 의미합니다.")
    else:
        print(">> 분석: Y축의 값이 크게 변합니다.")
        print(">> 결론: 도로에 경사가 있거나, 좌표계가 회전되어 있을 수 있습니다.")

else:
    print("[오류] 'road_positions' 변수가 없거나 데이터가 비어있습니다. 이전 단계 코드를 실행해주세요.")

Start printing all 610 points...
--------------------------------------------------
Index  | X (East/West) | Y (Height)   | Z (North/South)
--------------------------------------------------
0      | -185.13      | 0.00         | -209.85     
1      | -191.80      | 0.00         | -210.06     
2      | -193.33      | 0.00         | -201.11     
3      | -185.41      | 0.00         | -200.85     
4      | -202.10      | 0.00         | -213.30     
5      | -205.55      | 0.00         | -204.95     
6      | -206.96      | 0.00         | -215.84     
7      | -211.88      | 0.00         | -208.25     
8      | -210.16      | 0.00         | -218.38     
9      | -215.92      | 0.00         | -211.45     
10     | -212.14      | 0.00         | -220.10     
11     | -218.47      | 0.00         | -213.68     
12     | -220.54      | 0.00         | -229.50     
13     | -227.25      | 0.00         | -223.51     
14     | -222.55      | 0.00         | -231.75     
15     | -229.26      | 0.00 

In [ ]:
# ==============================================================================
# 3. 경로(Trajectory) 좌표 정밀 보정 (Raw Data Inspection)
# ==============================================================================
print("[보정] 빨간색 도로 좌표 정밀 분석 시작...")

road_positions = []
target_road_id = "elm__00"

if hasattr(scene, 'mi_scene'):
    for s in scene.mi_scene.shapes():
        if target_road_id in s.id():
            params = mi.traverse(s)
            if 'vertex_positions' in params:
                # 1) 원본 좌표 추출
                v_pos = np.array(params['vertex_positions'], dtype=np.float32).reshape(-1, 3)
                
                # 2) [진단] 412번 인덱스의 '원본(Raw)' 좌표 확인
                if len(v_pos) > 412:
                    raw_412 = v_pos[412]
                    print(f" -> [진단] Raw Index 412: {raw_412}")
                    # 예상: [418.xx, 0.0, -308.xx] 또는 [418.xx, -308.xx, 0.0] 등
                    
                    # 3) [해결] 목표 좌표(Target)와 비교하여 매핑 결정
                    # Target Z: -308.20
                    
                    # Case A: Raw Y가 -308 근처인 경우 -> Y를 Z로 (x, 0, y)
                    if np.isclose(raw_412[1], -308.2, atol=5.0):
                        print(" -> [결정] Y축 데이터를 Z축으로 이동합니다. (Y -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 1]
                        
                    # Case B: Raw Z가 -308 근처인 경우 -> 그대로 사용 (Z -> Z)
                    elif np.isclose(raw_412[2], -308.2, atol=5.0):
                        print(" -> [결정] Z축 데이터를 그대로 사용합니다. (No Rotation)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 2]
                        
                    # Case C: Raw Y가 308 근처인 경우 -> 부호 반전 후 이동 (-Y -> Z)
                    elif np.isclose(raw_412[1], 308.2, atol=5.0):
                        print(" -> [결정] Y축 데이터를 반전하여 Z축으로 이동합니다. (-Y -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = -v_pos[:, 1]

                    # Case D: Raw Z가 308 근처인 경우 -> 부호 반전 (-Z -> Z)
                    elif np.isclose(raw_412[2], 308.2, atol=5.0):
                        print(" -> [결정] Z축 데이터를 반전하여 사용합니다. (-Z -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = -v_pos[:, 2]
                        
                    else:
                        print(" -> [경고] 자동 매핑 실패. Raw 데이터가 예상과 다릅니다. 원본 그대로 사용합니다.")
                        # 기본: (x, 0, y) 시도 (가장 흔한 패턴)
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 1]

                print(f" -> 좌표 변환 완료 ({len(road_positions)} vertices)")
            break

In [ ]:
import os
import tensorflow as tf
import numpy as np
import drjit as dr
import gc
import sionna

# [버전 호환성] Import
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver

# ==========================================
# 1. 파일 저장 경로 및 기본 설정
# ==========================================
output_dir = "/data1/mh/sionna/tutorials/rt/build code/paper/L1_data"
if not os.path.exists(output_dir):
    try: os.makedirs(output_dir)
    except: output_dir = "."

# 저장할 파일명 (복소수 버전)
file_name = "vit_channel_dataset_precise_10s_complex_final.npy"
save_path = os.path.join(output_dir, file_name)

# Scene 경로
xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# ==========================================
# 2. 시스템 파라미터
# ==========================================
simulation_time = 10.0 # 10초
dt = 0.5e-3 # 0.5 ms
total_steps = int(simulation_time / dt)

carrier_frequency = 3.5e9
subcarrier_spacing = 30e3 
fft_size = 72
frequencies = subcarrier_spacing * tf.range(fft_size, dtype=tf.float32)

# 기지국 3개 위치
tx_positions = [[-125.663, 56.367, -181.453], [323.472, 36.869, -204.315], [0.663, 56.367, -181.453]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

# UE 설정 (속도별 8개)
speeds_kmh = [10, 15, 20, 25, 30, 40, 50, 60]
speeds_ms = [v / 3.6 for v in speeds_kmh]
num_ues = len(speeds_kmh)

print(f"--- 시뮬레이션 설정 ---")
print(f"총 시간: {simulation_time}초 ({total_steps} Steps)")
print(f"기지국 수: {len(tx_names)}개")
print(f"UE 수: {num_ues}명")
print(f"처리 방식: 1 Tx - 1 Rx 완전 순차 처리 (메모리 최적화)")

# ==========================================
# 3. 경로 및 Walker 설정
# ==========================================
if 'road_positions' not in globals():
    if os.path.exists("road_positions.npy"):
        road_positions = np.load("road_positions.npy")
    else:
        raise ValueError("메모리에 'road_positions' 변수가 없습니다.")

path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]
trajectory_points = road_positions[path_indices]

class PolylineWalker:
    def __init__(self, points):
        self.points = points
        diffs = points[1:] - points[:-1]
        self.seg_lengths = np.linalg.norm(diffs, axis=1)
        self.cum_dist = np.insert(np.cumsum(self.seg_lengths), 0, 0.0)
        self.total_length = self.cum_dist[-1]
    def get_position(self, distance):
        if distance >= self.total_length: return self.points[-1]
        if distance <= 0: return self.points[0]
        idx = np.searchsorted(self.cum_dist, distance) - 1
        idx = max(0, idx)
        p_start = self.points[idx]
        p_end = self.points[idx+1]
        ratio = (distance - self.cum_dist[idx]) / self.seg_lengths[idx] if self.seg_lengths[idx] > 0 else 0
        return p_start + (p_end - p_start) * ratio

walker = PolylineWalker(trajectory_points)

# ==========================================
# 4. Scene 구성 (빈 껍데기만 로드)
# ==========================================
scene = load_scene(temp_xml_path)

bs_array = PlanarArray(num_rows=8, num_cols=8, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="iso", polarization="V")
ue_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
scene.tx_array = bs_array
scene.rx_array = ue_array

solver = PathSolver()

# ==========================================
# 5. 시뮬레이션 실행 (완전 순차 처리)
# ==========================================
# 최종 데이터를 담을 리스트
all_ue_data = []

print(f"🚀 시뮬레이션 시작...")

for ue_idx in range(num_ues):
    ue_speed = speeds_ms[ue_idx]
    print(f"\n▶ [UE {ue_idx+1}/{num_ues}] (속도 {speeds_kmh[ue_idx]} km/h) 처리 시작...")
    
    # 현재 UE의 [BS1, BS2, BS3] 데이터를 담을 리스트
    current_ue_bs_data = [] 
    
    # 기지국 순차 처리
    for tx_idx, (tx_name, tx_pos) in enumerate(zip(tx_names, tx_positions)):
        print(f"  └─ BS {tx_idx+1}/{len(tx_names)} 계산 중...", end="")
        
        # 1. 맵 청소
        if "Active_Tx" in scene.transmitters: scene.remove("Active_Tx")
        if "Active_Rx" in scene.receivers: scene.remove("Active_Rx")
        
        # 2. 현재 Tx와 Rx 1개씩만 추가
        scene.add(Transmitter(name="Active_Tx", position=tx_pos, orientation=[0,0,0]))
        scene.add(Receiver(name="Active_Rx", position=trajectory_points[0], orientation=[0,0,0]))
        
        active_rx = scene.receivers["Active_Rx"]
        
        # 60초(10초) 데이터 담을 리스트
        time_step_data = []
        
        for step in range(total_steps):
            current_time = step * dt
            
            # [핵심 수정] 위치 이동 (지면 + 1.5m 높이)
            ground_pos = walker.get_position(ue_speed * current_time)
            new_pos = ground_pos + np.array([0, 0, 1.5]) # 높이 보정
            active_rx.position = new_pos
            
            # [핵심 수정] Ray Tracing (샘플 수 100,000개로 증가)
            paths = solver(scene, max_depth=3, samples_per_src=100000)
            
            # CFR 계산
            cfr_output = paths.cfr(frequencies=frequencies)
            
            if isinstance(cfr_output, tuple): h_freq = cfr_output[0]
            else: h_freq = cfr_output
                
            # NumPy 변환
            h_freq_np = h_freq.numpy()
            
            # [진단: 첫 스텝만]
            if step == 0 and ue_idx == 0 and tx_idx == 0:
                print(f"\n   [진단] Shape: {h_freq_np.shape}, Type: {h_freq_np.dtype}")
                # 에너지 확인
                total_energy = np.sum(np.abs(h_freq_np))
                print(f"   [진단] 감지된 신호 총량: {total_energy:.6f}")
                
                if total_energy == 0:
                    print("   ⚠️ [치명적 경고] 여전히 경로를 못 찾았습니다. 맵 좌표 단위를 확인해야 할 수도 있습니다.")
                else:
                    print("   ✅ [성공] 경로가 감지되었습니다.")

            # [핵심 수정] 복소수 합치기 (만약 분리된 경우)
            if not np.iscomplexobj(h_freq_np) and h_freq_np.shape[-1] == 2:
                h_freq_np = h_freq_np[..., 0] + 1j * h_freq_np[..., 1]
            
            # (1,1,1,64) -> (64,) 로 줄여서 저장
            time_step_data.append(np.squeeze(h_freq_np))
            
            # 메모리 정리
            del paths, cfr_output, h_freq
            if step % 2000 == 0:
                dr.flush_malloc_cache()
                gc.collect()

        # BS 하나 완료
        bs_np_data = np.array(time_step_data, dtype=np.complex64)
        current_ue_bs_data.append(bs_np_data)
        
        # 맵 청소
        scene.remove("Active_Tx")
        scene.remove("Active_Rx")
        dr.flush_malloc_cache()
        gc.collect()
        print(" 완료.")

    # UE 하나 완료: 데이터 임시 저장
    ue_combined = np.stack(current_ue_bs_data, axis=0) # (3, Time, 64)
    ue_combined = np.transpose(ue_combined, (1, 0, 2)) # (Time, 3, 64)
    
    all_ue_data.append(ue_combined)
    
    # 안전 백업
    np.save(os.path.join(output_dir, f"backup_UE_{ue_idx}.npy"), ue_combined)
    print(f"  💾 UE {ue_idx} 백업 완료.")

# ==========================================
# 6. 최종 병합 및 저장
# ==========================================
print("\n📊 데이터 병합 중...")
final_dataset = np.stack(all_ue_data, axis=0) # (UE, Time, BS, 64)
final_dataset = np.transpose(final_dataset, (1, 0, 2, 3)) # (Time, UE, BS, 64)

print(f"최종 데이터 형태: {final_dataset.shape}")
print(f"최종 데이터 타입: {final_dataset.dtype}")

if np.iscomplexobj(final_dataset):
    print("✅ 최종 확인: 복소수 데이터입니다.")
else:
    print("⚠️ 최종 확인: 실수 데이터입니다.")

np.save(save_path, final_dataset)
print(f"저장 완료: {save_path}")

--- 시뮬레이션 설정 ---
총 시간: 10.0초 (20000 Steps)
기지국 수: 3개
UE 수: 8명
처리 방식: 1 Tx - 1 Rx 완전 순차 처리 (메모리 최적화)
🚀 시뮬레이션 시작...

▶ [UE 1/8] (속도 10 km/h) 처리 시작...
  └─ BS 1/3 계산 중...
   [진단] Shape: (1, 1, 1, 64, 1, 72), Type: float32
   [진단] 감지된 신호 총량: 0.034772
   ✅ [성공] 경로가 감지되었습니다.


차량이 움직일 경로 설정

In [10]:
path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]

In [11]:
import os
import mitsuba as mi
from sionna.rt import load_scene

# ==============================================================================
# 1. XML 파일 경로 보정 및 로드
# ==============================================================================
# 원본 파일 경로 (사용자 환경에 맞게 설정)
original_xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(original_xml_path)
fixed_xml_path = os.path.join(scene_dir, "kookmin_fixed_temp.xml") # 임시 수정 파일

# 1) 원본 XML 읽기
with open(original_xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로로 치환하여 "파일 찾기 실패" 방지
#    (예: "meshes/file.ply" -> "/data1/mh/.../meshes/file.ply")
abs_mesh_path = os.path.join(scene_dir, "meshes") + "/"
xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')

# 3) 수정된 내용을 임시 파일로 저장
with open(fixed_xml_path, 'w', encoding='utf-8') as f:
    f.write(xml_content_fixed)

print(f"[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: {fixed_xml_path}")

# 4) 장면 로드
try:
    scene = load_scene(fixed_xml_path)
    print("[성공] 장면(Scene)을 성공적으로 불러왔습니다.")
except Exception as e:
    print(f"[오류] 장면 로드 실패: {e}")

# ==============================================================================
# 2. 재질(Material) 정보 확인
# ==============================================================================
if 'scene' in globals():
    print("\n" + "="*60)
    print(f"{'Object Name':<30} | {'Assigned Material':<20}")
    print("="*60)
    
    # 씬에 있는 모든 객체를 순회하며 할당된 재질 확인
    for name, obj in scene.objects.items():
        # 재질 객체가 있으면 이름 출력, 없으면 None
        mat_name = obj.radio_material.name if obj.radio_material else "None"
        print(f"{name:<30} | {mat_name:<20}")
        
    print("="*60)
    
    # 정의된 모든 재질(Radio Material) 목록 확인
    print("\n[정의된 재질 목록]")
    for mat_name, mat in scene.radio_materials.items():
        print(f" - 이름: {mat_name:<15} (Type: {mat.itu_type}, Thickness: {mat.thickness})")

[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_temp.xml
[성공] 장면(Scene)을 성공적으로 불러왔습니다.

Object Name                    | Assigned Material   
elm__23                        | wall                
elm__24                        | roof                
elm__25                        | 8b4513              
elm__26                        | 2f4f4f              
elm__27                        | red                 
elm__28                        | white               
elm__29                        | gray                
elm__30                        | black               
elm__31                        | darkgrey            
elm__32                        | grey                
elm__33                        | lightgrey           
elm__34                        | silver              
elm__35                        | brown               
elm__36                        | d2aa6d              
elm__37                        | a58e9a              
elm_

In [12]:
# ==============================================================================
# 1. XML 경로 수정 및 안전한 로드
# ==============================================================================
# 원본 파일 및 폴더 경로 (사용자 환경)
xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
meshes_dir = os.path.join(scene_dir, "meshes")
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# 1) XML 파일 읽기
with open(xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로("/data1/.../meshes/")로 치환
if os.path.exists(meshes_dir):
    abs_mesh_path = meshes_dir + "/" if not meshes_dir.endswith("/") else meshes_dir
    xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')
    
    # 3) 임시 파일로 저장
    with open(temp_xml_path, 'w', encoding='utf-8') as f:
        f.write(xml_content_fixed)
    print(f"[설정] 경로가 수정된 임시 XML 생성: {temp_xml_path}")
else:
    raise FileNotFoundError(f"meshes 폴더를 찾을 수 없습니다: {meshes_dir}")

# 4) load_scene으로 로드
try:
    scene = load_scene(temp_xml_path)
    print("[성공] 장면(Scene) 로드 완료.")
except Exception as e:
    print(f"[치명적 오류] 장면 로드 실패: {e}")
    raise e

#

[설정] 경로가 수정된 임시 XML 생성: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_absolute.xml
[성공] 장면(Scene) 로드 완료.


In [13]:
# ==============================================================================
# 2. 도로 재질 변경 (시각화용)
# ==============================================================================
red_road_mat = ITURadioMaterial(name="red_road_mat",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1.0, 0.0, 0.0])
scene.add(red_road_mat)

target_road_id = "elm__00"
if target_road_id in scene.objects:
    scene.objects[target_road_id].radio_material = red_road_mat
    print(f"[설정] 도로({target_road_id})를 빨간색으로 변경했습니다.")

[설정] 도로(elm__00)를 빨간색으로 변경했습니다.


In [14]:
# ==============================================================================
# 3. 경로(Trajectory) 좌표 추출 및 속도 설정
# ==============================================================================
road_positions = []

if hasattr(scene, 'mi_scene'):
    for s in scene.mi_scene.shapes():
        if target_road_id in s.id():
            params = mi.traverse(s)
            if 'vertex_positions' in params:
                v_pos = np.array(params['vertex_positions'], dtype=np.float32).reshape(-1, 3)
                
                # [좌표 회전] XML의 Transform (<rotate x="1" angle="-90"/>) 적용
                theta = np.radians(-90)
                c, s_sin = np.cos(theta), np.sin(theta)
                R_x = np.array([[1, 0, 0],
                                [0, c, -s_sin],
                                [0, s_sin, c]], dtype=np.float32)
                
                road_positions = v_pos @ R_x.T
                print(f"[데이터] 도로 좌표 {len(road_positions)}개 추출 완료.")
            break

if len(road_positions) == 0:
    print("[경고] 좌표 추출 실패. 임시 경로 생성.")
    t = np.linspace(0, 200, 100)
    road_positions = np.column_stack((t, np.zeros_like(t), t)).astype(np.float32)

# 사용자 지정 경로 인덱스
path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]

valid_indices = [i for i in path_indices if i < len(road_positions)]
path_coords = road_positions[valid_indices].copy()
path_coords[:, 1] += 1.5 

# 속도 벡터 계산 (60km/h)
speed_ms = 60.0 / 3.6 
diffs = path_coords[1:] - path_coords[:-1]
diffs = np.vstack([diffs, diffs[-1]])
norms = np.linalg.norm(diffs, axis=1, keepdims=True)
norms[norms == 0] = 1.0
velocities = (diffs / norms) * speed_ms

print(f"[설정] 총 {len(path_coords)}개 경로점, 속도 60km/h 설정 완료.")

[데이터] 도로 좌표 610개 추출 완료.
[설정] 총 60개 경로점, 속도 60km/h 설정 완료.


In [15]:
import tensorflow as tf
from sionna.rt import Transmitter, Receiver, PlanarArray

# ==============================================================================
# 4. 송수신기(Tx/Rx) 배치 (중복 오류 수정 및 API 수정)
# ==============================================================================

# 1) [수정] 기존 객체 확실하게 제거 (중복 방지 로직 개선)
# 기지국 제거
for name in ['Tx_1', 'Tx_2', 'Tx_3']:
    if name in scene.transmitters: # scene.objects가 아니라 scene.transmitters를 확인해야 함
        scene.remove(name)

# 수신기 제거
if 'rx_car' in scene.receivers: # scene.receivers 확인
    scene.remove('rx_car')

# 2) 안테나 패턴 정의
tx_array = PlanarArray(num_rows=1, num_cols=4, pattern="tr38901", polarization="VH")
rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
'''
# 3) 기지국(Tx) 배치
tx_indices = [340, 362, 522]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, idx in enumerate(tx_indices):
    if idx < len(road_positions):
        # 위치 계산
        pos = road_positions[idx].copy()
        pos[1] += 25.0 # 높이
        pos[0] += 15.0 # 이격
        pos[2] += 15.0
        
        # [수정] antenna 인자 제거 (생성 시 넣으면 에러남)
        tx = Transmitter(name=tx_names[i], 
                         position=pos, 
                         power_dbm=43)
        
        # [수정] 객체 생성 후 속성으로 할당
        tx.transmit_antenna = tx_array
        
        scene.add(tx)
        print(f"[배치] 기지국 {tx_names[i]} 설치 완료.")
'''
# 3) 기지국(Tx) 배치 - [사용자 지정 좌표]
# 요청하신 좌표 목록: (Tx_1, Tx_2, Tx_3 순서)
tx_positions = [
    [-125.663, 56.367, -181.453],  # Tx_1
    [323.472, 36.869, -204.315],   # Tx_2
    [0.663, 56.367, -181.453]   # Tx_3 (Tx_1과 동일)
]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    # 송신기 생성 및 안테나 할당
    tx = Transmitter(name=tx_names[i], 
                     position=pos, 
                     power_dbm=43)
    tx.transmit_antenna = tx_array
    
    scene.add(tx)
    print(f"[배치] 기지국 {tx_names[i]} 좌표 {pos}에 설치 완료.")

# 4) 차량(Rx) 배치
# 생성 시 antenna 인자 제거
rx = Receiver(name="rx_car", 
              position=path_coords[0])

# 객체 생성 후 속성으로 할당
rx.receive_antenna = rx_array

scene.add(rx)

# 5) 차량 경로 및 속도 할당 (NumPy Transpose 적용)
# [중요] (3, N) 형태로 입력해야 함
import numpy as np
rx.position = path_coords.astype(np.float32).T
rx.velocity = velocities.astype(np.float32).T

print("[배치] 차량 안테나 설정 및 경로 데이터 할당 완료.")

2026-01-30 15:24:37.035138: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769754277.078399   40708 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769754277.091025   40708 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769754277.171059   40708 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769754277.171071   40708 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769754277.171073   40708 computation_placer.cc:177] computation placer alr

[배치] 기지국 Tx_1 좌표 [-125.663, 56.367, -181.453]에 설치 완료.
[배치] 기지국 Tx_2 좌표 [323.472, 36.869, -204.315]에 설치 완료.
[배치] 기지국 Tx_3 좌표 [0.663, 56.367, -181.453]에 설치 완료.
[배치] 차량 안테나 설정 및 경로 데이터 할당 완료.


In [16]:

import numpy as np
import matplotlib.pyplot as plt
from sionna.rt import Camera, PathSolver, PlanarArray

# ==============================================================================
# 5. 시뮬레이션 및 시각화 (오류 수정됨)
# ==============================================================================

# 1) [핵심 수정] Scene 전체 안테나 설정
# PathSolver 실행을 위해 장면 전체의 기본 안테나를 명시해야 합니다.
# (이전 단계에서 정의한 안테나 배열과 동일하게 설정)
scene.tx_array = PlanarArray(num_rows=1, num_cols=4, pattern="tr38901", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

print("[설정] Scene 안테나 배열(tx_array, rx_array) 등록 완료.")

# 2) 카메라 위치 설정
center = np.mean(path_coords, axis=0)
cam = Camera(position=center + np.array([0, 500, 500]), look_at=center)

print(f"[연산] Ray Tracing 경로 계산을 시작합니다... (Sionna v1.2.1)")
print(f"       - 총 {len(path_coords)}개 지점 (Snapshots)")
print(f"       - 최대 반사 횟수: 5회")
print(f"       - 광선 샘플 수: 1,000,000개")

# 3) PathSolver 실행
try:
    # Solver 인스턴스 생성
    solver = PathSolver()
    
    # 경로 계산 (__call__)
    # scattering -> diffuse_reflection (문서 기준 수정)
    # num_samples -> samples_per_src (문서 기준 수정)
    paths = solver(scene, 
                   max_depth=5, 
                   samples_per_src=1000000, 
                   diffuse_reflection=False, 
                   diffraction=False)
    
    print("[완료] 경로 계산이 성공적으로 끝났습니다.")

except Exception as e:
    print(f"[오류] PathSolver 실행 실패: {e}")
    raise e

# 4) 3D 동적 시각화
print("[시각화] 3D 뷰어를 실행합니다.")
try:
    scene.preview(paths=paths, show_devices=True)
except Exception as e:
    print(f"[주의] 3D 뷰어 실행 오류: {e}")

# 5) CIR 데이터 분석 및 그래프 출력
try:
    # CIR 데이터 추출
    a, tau = paths.cir()

    # 분석할 스냅샷 (중간 지점)
    snap_idx = len(path_coords) // 2
    tx_idx = 0 

    # Tensor -> Numpy 변환
    amps = a[tx_idx, snap_idx, :].numpy()
    delays = tau[tx_idx, snap_idx, :].numpy() * 1e9 # ns 변환

    # 유효 신호 필터링
    valid_mask = np.abs(amps) > 0
    amps_valid = np.abs(amps[valid_mask])
    delays_valid = delays[valid_mask]

    # 그래프 그리기
    plt.figure(figsize=(10, 6))
    if len(amps_valid) > 0:
        # Stem plot
        markerline, stemlines, baseline = plt.stem(delays_valid, amps_valid, basefmt=" ")
        plt.setp(markerline, 'markerfacecolor', 'r')
        
        # 최대 신호 표시
        max_idx = np.argmax(amps_valid)
        plt.annotate(f'Strongest\n({delays_valid[max_idx]:.1f}ns)', 
                     xy=(delays_valid[max_idx], amps_valid[max_idx]),
                     xytext=(delays_valid[max_idx]+50, amps_valid[max_idx]),
                     arrowprops=dict(facecolor='black', shrink=0.05))
        
        plt.title(f"CIR Analysis: {tx_names[tx_idx]} -> Car (Snapshot {snap_idx})")
    else:
        plt.title(f"CIR Analysis: No Signal at Snapshot {snap_idx}")
        print("[결과] 해당 위치에서는 수신된 신호가 없습니다.")

    plt.xlabel("Delay (ns)")
    plt.ylabel("Amplitude |h|")
    plt.grid(True, alpha=0.3)
    plt.show()

except Exception as e:
    print(f"[오류] CIR 그래프 출력 실패: {e}")


[설정] Scene 안테나 배열(tx_array, rx_array) 등록 완료.
[연산] Ray Tracing 경로 계산을 시작합니다... (Sionna v1.2.1)
       - 총 60개 지점 (Snapshots)
       - 최대 반사 횟수: 5회
       - 광선 샘플 수: 1,000,000개
[완료] 경로 계산이 성공적으로 끝났습니다.
[시각화] 3D 뷰어를 실행합니다.


[오류] CIR 그래프 출력 실패: list indices must be integers or slices, not tuple


코드 수정


In [17]:
import numpy as np
from sionna.rt import Camera

# ==============================================================================
# [긴급 진단] 기지국 및 자동차 위치 시각화
# ==============================================================================

# 1. 카메라 설정 (전체 맵이 보이도록 높게 설정)
center = np.mean(path_coords, axis=0)
cam = Camera(position=center + np.array([0, 600, 600]), look_at=center)

print("="*60)
print("[진단 모드] 시뮬레이션 환경 시각화")
print("="*60)
print(" 1. 3D 뷰어가 뜨면 마우스로 회전/확대하여 '기지국(Tx)'과 '자동차(Rx)'를 찾으세요.")
print(" 2. 기지국이 건물 속에 파묻혀 있는지 확인하세요.")
print(" 3. 자동차 경로(점들)가 도로 위에 있는지 확인하세요.")
print("-" * 60)

# 2. 위치 좌표 출력 (텍스트 확인용)
print(f"Tx_1 좌표: {tx_positions[0]}")
print(f"Rx(차량) 첫 위치: {path_coords[0]}")
print(f"Rx(차량) 평균 높이: {np.mean(path_coords[:, 1]):.2f} m")

# 3. 뷰어 실행 (경로 계산 없이 기기 위치만 표시)
# show_devices=True: 기지국과 수신기 위치를 아이콘으로 표시함
# resolution: 해상도 조절 (필요시)
try:
    scene.preview(show_devices=True, resolution=[800, 600])
except Exception as e:
    print(f"[오류] 뷰어 실행 실패: {e}")

[진단 모드] 시뮬레이션 환경 시각화
 1. 3D 뷰어가 뜨면 마우스로 회전/확대하여 '기지국(Tx)'과 '자동차(Rx)'를 찾으세요.
 2. 기지국이 건물 속에 파묻혀 있는지 확인하세요.
 3. 자동차 경로(점들)가 도로 위에 있는지 확인하세요.
------------------------------------------------------------
Tx_1 좌표: [-125.663, 56.367, -181.453]
Rx(차량) 첫 위치: [ 4.1827621e+02 -3.0670306e+02 -3.7743991e-14]
Rx(차량) 평균 높이: -224.62 m


In [18]:
# ==============================================================================
# 3. 경로(Trajectory) 좌표 추출 및 속도 설정 (좌표축 오류 수정됨)
# ==============================================================================
import numpy as np
import mitsuba as mi

road_positions = []

if hasattr(scene, 'mi_scene'):
    for s in scene.mi_scene.shapes():
        if target_road_id in s.id():
            params = mi.traverse(s)
            if 'vertex_positions' in params:
                # 1) 원본 좌표 추출
                v_pos = np.array(params['vertex_positions'], dtype=np.float32).reshape(-1, 3)
                
                # [진단 및 수정]
                # 기존 회전 코드가 평면 좌표를 높이(Y)로 보내버리는 문제가 있었습니다.
                # 회전 행렬을 제거하고, 좌표축을 직접 재할당하여 눕힙니다.
                
                # X축: 그대로 사용
                # Y축: 원본 데이터의 Y축 -> Z축으로 매핑 (또는 상황에 따라 원본 Z -> Y)
                # 여기서는 데이터를 '강제로' XZ 평면에 맞춥니다.
                
                # 임시 배열 생성
                fixed_pos = np.zeros_like(v_pos)
                
                # X는 그대로
                fixed_pos[:, 0] = v_pos[:, 0]
                
                # [핵심] Y축(높이)에 있는 큰 값들을 Z축(남북)으로 보냄
                # (이전 결과에서 Y가 -300이었으므로, 이를 Z로 보냅니다)
                # 원본 데이터가 (x, y, 0) 형태였다면 -> (x, 0, y)로 바꿈
                fixed_pos[:, 2] = v_pos[:, 1] 
                
                # 높이(Y)는 일단 0으로 초기화 (나중에 1.5m 더함)
                fixed_pos[:, 1] = 0.0
                
                # 만약 Z축 방향이 반대라면 -1을 곱해줍니다. (필요시 주석 해제)
                # fixed_pos[:, 2] *= -1 
                
                road_positions = fixed_pos
                print(f"[데이터] 도로 좌표 {len(road_positions)}개 추출 및 축 보정 완료.")
            break

if len(road_positions) == 0:
    print("[경고] 좌표 추출 실패. 임시 경로 생성.")
    t = np.linspace(0, 200, 100)
    road_positions = np.column_stack((t, np.zeros_like(t), t)).astype(np.float32)

# 사용자 지정 경로 인덱스
path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]

# 유효 인덱스 필터링
valid_indices = [i for i in path_indices if i < len(road_positions)]
path_coords = road_positions[valid_indices].copy()

# [높이 설정] 지면(0m) + 차량 높이(1.5m)
# 만약 건물이 떠 있다면 0.0 대신 10.0 등으로 조절 가능
path_coords[:, 1] = 0.0 + 1.5 

# 속도 벡터 계산 (60km/h)
speed_ms = 60.0 / 3.6 
diffs = path_coords[1:] - path_coords[:-1]
diffs = np.vstack([diffs, diffs[-1]])
norms = np.linalg.norm(diffs, axis=1, keepdims=True)
norms[norms == 0] = 1.0
velocities = (diffs / norms) * speed_ms

print(f"[설정] 경로 {len(path_coords)}개 수정 완료.")
print(f"      -> 차량 높이(Y) 강제 설정: {path_coords[0, 1]} m")
print(f"      -> 첫 위치: {path_coords[0]}")

[데이터] 도로 좌표 610개 추출 및 축 보정 완료.
[설정] 경로 60개 수정 완료.
      -> 차량 높이(Y) 강제 설정: 1.5 m
      -> 첫 위치: [4.1827621e+02 1.5000000e+00 1.8871996e-14]


In [19]:
import tensorflow as tf
from sionna.rt import Transmitter, Receiver, PlanarArray

# ==============================================================================
# 4. 송수신기(Tx/Rx) 배치 (중복 오류 수정 및 API 수정)
# ==============================================================================

# 1) [수정] 기존 객체 확실하게 제거 (중복 방지 로직 개선)
# 기지국 제거
for name in ['Tx_1', 'Tx_2', 'Tx_3']:
    if name in scene.transmitters: # scene.objects가 아니라 scene.transmitters를 확인해야 함
        scene.remove(name)

# 수신기 제거
if 'rx_car' in scene.receivers: # scene.receivers 확인
    scene.remove('rx_car')

# 2) 안테나 패턴 정의
tx_array = PlanarArray(num_rows=1, num_cols=4, pattern="tr38901", polarization="VH")
rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
'''
# 3) 기지국(Tx) 배치
tx_indices = [340, 362, 522]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, idx in enumerate(tx_indices):
    if idx < len(road_positions):
        # 위치 계산
        pos = road_positions[idx].copy()
        pos[1] += 25.0 # 높이
        pos[0] += 15.0 # 이격
        pos[2] += 15.0
        
        # [수정] antenna 인자 제거 (생성 시 넣으면 에러남)
        tx = Transmitter(name=tx_names[i], 
                         position=pos, 
                         power_dbm=43)
        
        # [수정] 객체 생성 후 속성으로 할당
        tx.transmit_antenna = tx_array
        
        scene.add(tx)
        print(f"[배치] 기지국 {tx_names[i]} 설치 완료.")
'''
# 3) 기지국(Tx) 배치 - [사용자 지정 좌표]
# 요청하신 좌표 목록: (Tx_1, Tx_2, Tx_3 순서)
tx_positions = [
    [-125.663, 56.367, -181.453],  # Tx_1
    [323.472, 36.869, -204.315],   # Tx_2
    [0.663, 56.367, -181.453]   # Tx_3 
]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    # 송신기 생성 및 안테나 할당
    tx = Transmitter(name=tx_names[i], 
                     position=pos, 
                     power_dbm=43)
    tx.transmit_antenna = tx_array
    
    scene.add(tx)
    print(f"[배치] 기지국 {tx_names[i]} 좌표 {pos}에 설치 완료.")

# 4) 차량(Rx) 배치
# 생성 시 antenna 인자 제거
rx = Receiver(name="rx_car", 
              position=path_coords[0])

# 객체 생성 후 속성으로 할당
rx.receive_antenna = rx_array

scene.add(rx)

# 5) 차량 경로 및 속도 할당 (NumPy Transpose 적용)
# [중요] (3, N) 형태로 입력해야 함
import numpy as np
rx.position = path_coords.astype(np.float32).T
rx.velocity = velocities.astype(np.float32).T

print("[배치] 차량 안테나 설정 및 경로 데이터 할당 완료.")

[배치] 기지국 Tx_1 좌표 [-125.663, 56.367, -181.453]에 설치 완료.
[배치] 기지국 Tx_2 좌표 [323.472, 36.869, -204.315]에 설치 완료.
[배치] 기지국 Tx_3 좌표 [0.663, 56.367, -181.453]에 설치 완료.
[배치] 차량 안테나 설정 및 경로 데이터 할당 완료.


In [20]:

import numpy as np
import matplotlib.pyplot as plt
from sionna.rt import Camera, PathSolver, PlanarArray

# ==============================================================================
# 5. 시뮬레이션 및 시각화 (오류 수정됨)
# ==============================================================================

# 1) [핵심 수정] Scene 전체 안테나 설정
# PathSolver 실행을 위해 장면 전체의 기본 안테나를 명시해야 합니다.
# (이전 단계에서 정의한 안테나 배열과 동일하게 설정)
scene.tx_array = PlanarArray(num_rows=1, num_cols=4, pattern="tr38901", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

print("[설정] Scene 안테나 배열(tx_array, rx_array) 등록 완료.")

# 2) 카메라 위치 설정
center = np.mean(path_coords, axis=0)
cam = Camera(position=center + np.array([0, 500, 500]), look_at=center)

print(f"[연산] Ray Tracing 경로 계산을 시작합니다... (Sionna v1.2.1)")
print(f"       - 총 {len(path_coords)}개 지점 (Snapshots)")
print(f"       - 최대 반사 횟수: 5회")
print(f"       - 광선 샘플 수: 1,000,000개")

# 3) PathSolver 실행
try:
    # Solver 인스턴스 생성
    solver = PathSolver()
    
    # 경로 계산 (__call__)
    # scattering -> diffuse_reflection (문서 기준 수정)
    # num_samples -> samples_per_src (문서 기준 수정)
    paths = solver(scene, 
                   max_depth=5, 
                   samples_per_src=1000000, 
                   diffuse_reflection=False, 
                   diffraction=False)
    
    print("[완료] 경로 계산이 성공적으로 끝났습니다.")

except Exception as e:
    print(f"[오류] PathSolver 실행 실패: {e}")
    raise e

# 4) 3D 동적 시각화
print("[시각화] 3D 뷰어를 실행합니다.")
try:
    scene.preview(paths=paths, show_devices=True)
except Exception as e:
    print(f"[주의] 3D 뷰어 실행 오류: {e}")

# 5) CIR 데이터 분석 및 그래프 출력
try:
    # CIR 데이터 추출
    a, tau = paths.cir()

    # 분석할 스냅샷 (중간 지점)
    snap_idx = len(path_coords) // 2
    tx_idx = 0 

    # Tensor -> Numpy 변환
    amps = a[tx_idx, snap_idx, :].numpy()
    delays = tau[tx_idx, snap_idx, :].numpy() * 1e9 # ns 변환

    # 유효 신호 필터링
    valid_mask = np.abs(amps) > 0
    amps_valid = np.abs(amps[valid_mask])
    delays_valid = delays[valid_mask]

    # 그래프 그리기
    plt.figure(figsize=(10, 6))
    if len(amps_valid) > 0:
        # Stem plot
        markerline, stemlines, baseline = plt.stem(delays_valid, amps_valid, basefmt=" ")
        plt.setp(markerline, 'markerfacecolor', 'r')
        
        # 최대 신호 표시
        max_idx = np.argmax(amps_valid)
        plt.annotate(f'Strongest\n({delays_valid[max_idx]:.1f}ns)', 
                     xy=(delays_valid[max_idx], amps_valid[max_idx]),
                     xytext=(delays_valid[max_idx]+50, amps_valid[max_idx]),
                     arrowprops=dict(facecolor='black', shrink=0.05))
        
        plt.title(f"CIR Analysis: {tx_names[tx_idx]} -> Car (Snapshot {snap_idx})")
    else:
        plt.title(f"CIR Analysis: No Signal at Snapshot {snap_idx}")
        print("[결과] 해당 위치에서는 수신된 신호가 없습니다.")

    plt.xlabel("Delay (ns)")
    plt.ylabel("Amplitude |h|")
    plt.grid(True, alpha=0.3)
    plt.show()

except Exception as e:
    print(f"[오류] CIR 그래프 출력 실패: {e}")


[설정] Scene 안테나 배열(tx_array, rx_array) 등록 완료.
[연산] Ray Tracing 경로 계산을 시작합니다... (Sionna v1.2.1)
       - 총 60개 지점 (Snapshots)
       - 최대 반사 횟수: 5회
       - 광선 샘플 수: 1,000,000개
[완료] 경로 계산이 성공적으로 끝났습니다.
[시각화] 3D 뷰어를 실행합니다.


[오류] CIR 그래프 출력 실패: list indices must be integers or slices, not tuple


In [21]:
import numpy as np
import mitsuba as mi
import tensorflow as tf
from sionna.rt import Camera, PathSolver, PlanarArray

# ==============================================================================
# 3. 경로(Trajectory) 좌표 재추출 (Big Table 값으로 복원)
# ==============================================================================
print("[보정] 빨간색 도로 좌표 정밀 복원 시작...")

road_positions = []
target_road_id = "elm__00"

if hasattr(scene, 'mi_scene'):
    for s in scene.mi_scene.shapes():
        if target_road_id in s.id():
            params = mi.traverse(s)
            if 'vertex_positions' in params:
                # 1) 원본 좌표 추출
                v_pos = np.array(params['vertex_positions'], dtype=np.float32).reshape(-1, 3)
                
                # 2) [복원] XML의 Transform (<rotate x="1" angle="-90"/>) 정확히 적용
                # 이 행렬이 적용되어야 사용자님이 보신 "Big Table"의 값이 나옵니다.
                # (x, y, z) -> (x, z, -y)
                theta = np.radians(-90)
                c, s_sin = np.cos(theta), np.sin(theta)
                R_x = np.array([[1, 0, 0],
                                [0, c, -s_sin],
                                [0, s_sin, c]], dtype=np.float32)
                
                # 좌표 변환
                road_positions = v_pos @ R_x.T
                
                print(f" -> 좌표 변환 완료 ({len(road_positions)} vertices)")
                
                # [검증] 412번 좌표 확인 (사용자 요청 값: 418.28, 0.00, -308.20)
                if len(road_positions) > 412:
                    chk = road_positions[412]
                    print(f" -> [검증] Index 412 좌표: [{chk[0]:.2f}, {chk[1]:.2f}, {chk[2]:.2f}]")
                    if np.isclose(chk[0], 418.28, atol=1.0):
                        print("    >> 확인: 사용자 요청 좌표와 일치합니다! (정상)")
                    else:
                        print("    >> 경고: 좌표가 예상과 다릅니다. 원본 데이터가 변경되었을 수 있습니다.")
            break

# 좌표가 없으면 비상용 경로
if len(road_positions) == 0:
    print("[비상] 좌표 추출 실패. 직선 경로 사용.")
    road_positions = np.array([[0,0,0], [100,0,0]], dtype=np.float32)

# ------------------------------------------------------------------------------
# 사용자 지정 경로 인덱스 적용
# ------------------------------------------------------------------------------
path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]

# 유효 인덱스만 필터링
valid_indices = [i for i in path_indices if i < len(road_positions)]
path_coords = road_positions[valid_indices].copy()

# 높이 보정 (1.5m)
path_coords[:, 1] += 1.5 

# 속도 및 방향 계산
speed_ms = 60.0 / 3.6 
diffs = path_coords[1:] - path_coords[:-1]
diffs = np.vstack([diffs, diffs[-1]])
norms = np.linalg.norm(diffs, axis=1, keepdims=True)
norms[norms == 0] = 1.0
velocities = (diffs / norms) * speed_ms

print(f"[완료] 최종 경로 설정: {len(path_coords)}개 점.")
print(f" -> 시작점(412): {path_coords[0]}")
print(f" -> 끝점(582):   {path_coords[-1]}")

# ==============================================================================
# 4. 송수신기 재배치 및 시뮬레이션 (자동 실행)
# ==============================================================================
# 기존 객체 제거
for name in ['Tx_1', 'Tx_2', 'Tx_3']:
    if name in scene.transmitters: scene.remove(name)
if 'rx_car' in scene.receivers: scene.remove('rx_car')

# 안테나 설정
scene.tx_array = PlanarArray(num_rows=1, num_cols=4, pattern="tr38901", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

# 기지국 재배치 (요청 좌표)
tx_positions = [
    [-125.663, 56.367, -181.453],
    [323.472, 36.869, -204.315], 
    [0.663, 56.367, -181.453]
]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    scene.add(tx)

# 차량 재배치
rx = Receiver(name="rx_car", position=path_coords[0])
rx.receive_antenna = scene.rx_array
scene.add(rx)

# 경로 할당 (Transpose 적용)
rx.position = path_coords.T
rx.velocity = velocities.T

print("[설정] 기지국 및 차량 재배치 완료.")

# 시뮬레이션 실행 (PathSolver)
print("[연산] Ray Tracing 다시 시작...")
solver = PathSolver()
paths = solver(scene, max_depth=5, samples_per_src=1000000, diffuse_reflection=True, diffraction=True)

# 결과 확인
print("[시각화] 3D 뷰어 실행 (이제 차가 빨간 도로 위에 있어야 합니다)")
scene.preview(paths=paths, show_devices=True)

[보정] 빨간색 도로 좌표 정밀 복원 시작...
 -> 좌표 변환 완료 (610 vertices)
 -> [검증] Index 412 좌표: [418.28, -308.20, -0.00]
    >> 확인: 사용자 요청 좌표와 일치합니다! (정상)
[완료] 최종 경로 설정: 60개 점.
 -> 시작점(412): [ 4.1827621e+02 -3.0670306e+02 -3.7743991e-14]
 -> 끝점(582):   [-2.2164143e+02 -1.6403864e+02 -2.0272638e-14]
[설정] 기지국 및 차량 재배치 완료.
[연산] Ray Tracing 다시 시작...
[시각화] 3D 뷰어 실행 (이제 차가 빨간 도로 위에 있어야 합니다)


In [22]:
import numpy as np

# ==============================================================================
# 자동차 위치 정밀 검증 (Index 412)
# ==============================================================================

# 1. 412번의 기대 좌표 (사용자 제공 값)
expected_pos_412 = np.array([418.28, 0.00, -308.20])

# 2. 현재 설정된 자동차의 시작 위치 (path_coords의 첫 번째 점)
#    (높이 보정 1.5m가 적용되어 있는지 확인 필요)
current_start_pos = path_coords[0].copy()

# 높이 보정(1.5m)을 제외하고 비교하기 위해 Y값을 0으로 맞춤
current_pos_flat = current_start_pos.copy()
current_pos_flat[1] = 0.0 

print("="*60)
print("[위치 검증] 자동차가 412번 좌표에 있는가?")
print("="*60)
print(f"1. 412번 목표 좌표 : {expected_pos_412}")
print(f"2. 자동차 시작 위치: {current_start_pos} (높이 포함)")
print(f"3. 자동차 평면 위치: {current_pos_flat} (높이 제외)")
print("-" * 60)

# 3. 오차 계산 (허용 오차 1.0m 이내)
diff = np.linalg.norm(current_pos_flat - expected_pos_412)

if diff < 1.0:
    print(f"✅ [성공] 일치합니다! (오차: {diff:.4f} m)")
    print("   -> 자동차가 412번 좌표에서 출발 준비를 마쳤습니다.")
else:
    print(f"❌ [실패] 위치가 다릅니다. (오차: {diff:.4f} m)")
    print("   -> 'road_positions' 좌표 변환 로직을 다시 확인해야 합니다.")
    print("   -> 412번 좌표가 추출되지 않았거나, 순서가 섞였을 수 있습니다.")

# 4. (선택) 412번 위치에 시각적 마커(Marker) 설치
#    뷰어에서 빨간색 공(Transmitter)으로 표시하여 눈으로 확인
try:
    # 기존 마커 제거
    if "Marker_412" in scene.transmitters: scene.remove("Marker_412")
    
    # 412번 위치에 임시 송신기 배치 (눈에 잘 띄게 높이 5m)
    marker_pos = expected_pos_412.copy()
    marker_pos[1] += 5.0 
    
    # 송신기 추가 (안테나는 기존 것 사용)
    marker = Transmitter(name="Marker_412", position=marker_pos, power_dbm=0)
    marker.transmit_antenna = scene.tx_array
    scene.add(marker)
    
    print("\n[시각화] 412번 위치에 'Marker_412'를 표시했습니다.")
    print("       3D 뷰어에서 자동차(Rx)가 이 마커 바로 아래에 있는지 확인하세요.")
except Exception as e:
    print(f"[시각화 오류] 마커 설치 실패: {e}")

[위치 검증] 자동차가 412번 좌표에 있는가?
1. 412번 목표 좌표 : [ 418.28    0.   -308.2 ]
2. 자동차 시작 위치: [ 4.1827621e+02 -3.0670306e+02 -3.7743991e-14] (높이 포함)
3. 자동차 평면 위치: [ 4.182762e+02  0.000000e+00 -3.774399e-14] (높이 제외)
------------------------------------------------------------
❌ [실패] 위치가 다릅니다. (오차: 308.2000 m)
   -> 'road_positions' 좌표 변환 로직을 다시 확인해야 합니다.
   -> 412번 좌표가 추출되지 않았거나, 순서가 섞였을 수 있습니다.

[시각화] 412번 위치에 'Marker_412'를 표시했습니다.
       3D 뷰어에서 자동차(Rx)가 이 마커 바로 아래에 있는지 확인하세요.


In [23]:
import numpy as np
import mitsuba as mi
import tensorflow as tf
from sionna.rt import Camera, PathSolver, PlanarArray

# ==============================================================================
# 3. 경로(Trajectory) 좌표 정밀 보정 (Raw Data Inspection)
# ==============================================================================
print("[보정] 빨간색 도로 좌표 정밀 분석 시작...")

road_positions = []
target_road_id = "elm__00"

if hasattr(scene, 'mi_scene'):
    for s in scene.mi_scene.shapes():
        if target_road_id in s.id():
            params = mi.traverse(s)
            if 'vertex_positions' in params:
                # 1) 원본 좌표 추출
                v_pos = np.array(params['vertex_positions'], dtype=np.float32).reshape(-1, 3)
                
                # 2) [진단] 412번 인덱스의 '원본(Raw)' 좌표 확인
                if len(v_pos) > 412:
                    raw_412 = v_pos[412]
                    print(f" -> [진단] Raw Index 412: {raw_412}")
                    # 예상: [418.xx, 0.0, -308.xx] 또는 [418.xx, -308.xx, 0.0] 등
                    
                    # 3) [해결] 목표 좌표(Target)와 비교하여 매핑 결정
                    # Target Z: -308.20
                    
                    # Case A: Raw Y가 -308 근처인 경우 -> Y를 Z로 (x, 0, y)
                    if np.isclose(raw_412[1], -308.2, atol=5.0):
                        print(" -> [결정] Y축 데이터를 Z축으로 이동합니다. (Y -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 1]
                        
                    # Case B: Raw Z가 -308 근처인 경우 -> 그대로 사용 (Z -> Z)
                    elif np.isclose(raw_412[2], -308.2, atol=5.0):
                        print(" -> [결정] Z축 데이터를 그대로 사용합니다. (No Rotation)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 2]
                        
                    # Case C: Raw Y가 308 근처인 경우 -> 부호 반전 후 이동 (-Y -> Z)
                    elif np.isclose(raw_412[1], 308.2, atol=5.0):
                        print(" -> [결정] Y축 데이터를 반전하여 Z축으로 이동합니다. (-Y -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = -v_pos[:, 1]

                    # Case D: Raw Z가 308 근처인 경우 -> 부호 반전 (-Z -> Z)
                    elif np.isclose(raw_412[2], 308.2, atol=5.0):
                        print(" -> [결정] Z축 데이터를 반전하여 사용합니다. (-Z -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = -v_pos[:, 2]
                        
                    else:
                        print(" -> [경고] 자동 매핑 실패. Raw 데이터가 예상과 다릅니다. 원본 그대로 사용합니다.")
                        # 기본: (x, 0, y) 시도 (가장 흔한 패턴)
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 1]

                print(f" -> 좌표 변환 완료 ({len(road_positions)} vertices)")
            break

if len(road_positions) == 0:
    print("[비상] 좌표 추출 실패.")
    road_positions = np.array([[0,0,0]], dtype=np.float32)

# ------------------------------------------------------------------------------
# 사용자 지정 경로 적용
# ------------------------------------------------------------------------------
path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]

# 인덱스 필터링
valid_indices = [i for i in path_indices if i < len(road_positions)]
path_coords = road_positions[valid_indices].copy()

# 높이 보정
path_coords[:, 1] += 1.5 

# 속도 재계산
speed_ms = 60.0 / 3.6 
diffs = path_coords[1:] - path_coords[:-1]
diffs = np.vstack([diffs, diffs[-1]])
norms = np.linalg.norm(diffs, axis=1, keepdims=True)
norms[norms == 0] = 1.0
velocities = (diffs / norms) * speed_ms

print(f"[검증] 412번 변환 결과: {path_coords[0]}")
# 목표: [418.28, 1.5, -308.20]

# ==============================================================================
# 4. 송수신기 재배치 및 시뮬레이션
# ==============================================================================
# 객체 리셋
for name in ['Tx_1', 'Tx_2', 'Tx_3']:
    if name in scene.transmitters: scene.remove(name)
if 'rx_car' in scene.receivers: scene.remove('rx_car')

# 안테나 설정
scene.tx_array = PlanarArray(num_rows=1, num_cols=4, pattern="tr38901", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

# 기지국 (요청 좌표)
tx_positions = [
    [-125.663, 56.367, -181.453],
    [323.472, 36.869, -204.315], 
    [0.663, 56.367, -181.453]
]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    scene.add(tx)

# 차량
rx = Receiver(name="rx_car", position=path_coords[0])
rx.receive_antenna = scene.rx_array
scene.add(rx)

# 경로 할당
rx.position = path_coords.T
rx.velocity = velocities.T

print("[설정] 재배치 완료. 시뮬레이션 시작...")

# 시뮬레이션
solver = PathSolver()
paths = solver(scene, max_depth=5, samples_per_src=1000000, diffuse_reflection=True, diffraction=True)

# 결과 확인
print("[시각화] 3D 뷰어 실행")
scene.preview(paths=paths, show_devices=True)

[보정] 빨간색 도로 좌표 정밀 분석 시작...
 -> [진단] Raw Index 412: [ 4.1827621e+02  1.8871996e-14 -3.0820306e+02]
 -> [결정] Z축 데이터를 그대로 사용합니다. (No Rotation)
 -> 좌표 변환 완료 (610 vertices)
[검증] 412번 변환 결과: [ 418.2762     1.5     -308.20306]
[설정] 재배치 완료. 시뮬레이션 시작...
[시각화] 3D 뷰어 실행


In [24]:
import numpy as np
import mitsuba as mi
import tensorflow as tf
from sionna.rt import Camera, PathSolver, PlanarArray, Transmitter, Receiver, ITURadioMaterial, SceneObject

# ==============================================================================
# 3. 경로(Trajectory) 좌표 재추출 및 검증 (회전 로직 수정)
# ==============================================================================
print("[보정] 빨간색 도로 좌표 정밀 복원 시작...")

road_positions = []
target_road_id = "elm__00"

if hasattr(scene, 'mi_scene'):
    for s in scene.mi_scene.shapes():
        if target_road_id in s.id():
            params = mi.traverse(s)
            if 'vertex_positions' in params:
                # 1) 원본 좌표 추출
                v_pos = np.array(params['vertex_positions'], dtype=np.float32).reshape(-1, 3)
                
                # [진단] 412번 인덱스의 원본 좌표 확인
                # target: [418.28, 0.00, -308.20]
                if len(v_pos) > 412:
                    raw_412 = v_pos[412]
                    print(f" -> Raw v_pos[412]: {raw_412}")
                    
                    # [판단 로직]
                    # Case A: Raw data matches target -> Use as is.
                    if np.isclose(raw_412[0], 418.28, atol=5.0) and np.isclose(raw_412[2], -308.20, atol=5.0):
                        print(" -> 원본 데이터가 이미 올바른 좌표계입니다. 회전 없이 사용합니다.")
                        road_positions = v_pos
                        
                    # Case B: Raw data is (418, 308, 0) -> Need Rotation
                    elif np.isclose(raw_412[0], 418.28, atol=5.0) and np.isclose(raw_412[1], 308.20, atol=5.0):
                         print(" -> 데이터가 XY 평면에 있습니다. -90도 회전을 적용합니다.")
                         theta = np.radians(-90)
                         c, s = np.cos(theta), np.sin(theta)
                         R_x = np.array([[1, 0, 0], [0, c, -s], [0, s, c]], dtype=np.float32)
                         road_positions = v_pos @ R_x.T
                         
                    # Case C: Other -> Use raw (fallback)
                    else:
                        print(" -> 좌표가 예상과 다르지만, 원본을 그대로 사용합니다 (Z축 데이터 보존).")
                        road_positions = v_pos
                        
                else:
                    road_positions = v_pos

            break

# 좌표가 없으면 비상용 경로
if len(road_positions) == 0:
    print("[비상] 좌표 추출 실패. 직선 경로 사용.")
    road_positions = np.array([[0,0,0], [100,0,0]], dtype=np.float32)

# ------------------------------------------------------------------------------
# 사용자 지정 경로 인덱스 적용
# ------------------------------------------------------------------------------
path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]

# 유효 인덱스만 필터링
valid_indices = [i for i in path_indices if i < len(road_positions)]
path_coords = road_positions[valid_indices].copy()

# 높이 보정 (1.5m)
path_coords[:, 1] += 1.5 

# 속도 및 방향 계산
speed_ms = 60.0 / 3.6 
diffs = path_coords[1:] - path_coords[:-1]
diffs = np.vstack([diffs, diffs[-1]])
norms = np.linalg.norm(diffs, axis=1, keepdims=True)
norms[norms == 0] = 1.0
velocities = (diffs / norms) * speed_ms

print(f"[완료] 최종 경로 설정: {len(path_coords)}개 점.")
print(f" -> 시작점(412): {path_coords[0]}")

# ==============================================================================
# 4. 송수신기 재배치 & 녹색 자동차 마커 추가
# ==============================================================================
# 기존 객체 제거
if 'rx_car' in scene.receivers: scene.remove('rx_car')
if 'Green_Car_Marker' in scene.objects: scene.remove('Green_Car_Marker') # 기존 마커 제거

# 안테나 설정
scene.tx_array = PlanarArray(num_rows=1, num_cols=4, pattern="tr38901", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

# 기지국 재배치 (요청 좌표)
tx_positions = [
    [-125.663, 56.367, -181.453],
    [323.472, 36.869, -204.315], 
    [0.663, 56.367, -181.453]
]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    # 중복 방지
    if tx_names[i] in scene.transmitters: scene.remove(tx_names[i])
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    scene.add(tx)

# 차량(Rx) 재배치
rx = Receiver(name="rx_car", position=path_coords[0])
rx.receive_antenna = scene.rx_array
scene.add(rx)

# 경로 할당 (Transpose 적용)
rx.position = path_coords.T
rx.velocity = velocities.T

print("[설정] 기지국 및 차량 재배치 완료.")

# [추가 요청] 자동차를 초록색으로 표시하고 크기를 키움
# Receiver 아이콘 자체는 변경할 수 없으므로, 'SceneObject'를 사용하여 시각적 마커를 추가합니다.
# (이 마커는 고정되어 있지만, 시작 위치를 명확히 보여줍니다)
try:
    # 녹색 재질 생성
    green_mat = ITURadioMaterial(name="green_car_mat",
                                 itu_type="metal",
                                 color=[0.0, 1.0, 0.0]) # Green
    if "green_car_mat" not in scene.radio_materials:
        scene.add(green_mat)
    
    # 큐브(Cube) 객체 추가 (크기는 scale로 조절)
    # Sionna 기본 큐브 로드 (없으면 생략)
    # 대안: Transmitter를 추가하되, 안테나 패턴을 시각화용으로 사용? -> SceneObject가 나음.
    # 하지만 내장 primitive가 없으므로, 가장 간단한 방법은 'Transmitter'를 하나 더 추가하는 것입니다.
    # 뷰어에서 Transmitter는 보통 다른 색상으로 표시됩니다.
    pass 
    
    # [대안] 시작점에 'Big Green Marker' (Transmitter) 추가
    # 뷰어에서 Rx와 구별되도록 높이를 약간 올림
    marker_pos = path_coords[0].copy()
    marker_pos[1] += 3.0
    
    if "Car_Marker" in scene.transmitters: scene.remove("Car_Marker")
    marker = Transmitter(name="Car_Marker", position=marker_pos, power_dbm=0)
    marker.transmit_antenna = scene.tx_array
    scene.add(marker)
    print("[시각화] 자동차 위치에 마커(Car_Marker)를 추가했습니다.")

except Exception as e:
    print(f"[시각화 설정 오류] {e}")

# ==============================================================================
# 5. 시뮬레이션 및 검증
# ==============================================================================
# 위치 검증
diff = np.linalg.norm(path_coords[0] - np.array([418.28, 1.5, -308.20]))
if diff < 2.0:
    print(f"✅ [성공] 자동차가 목표 좌표(412번)에 정확히 위치합니다. (오차: {diff:.2f}m)")
else:
    print(f"❌ [경고] 위치 오차가 여전히 존재합니다. (오차: {diff:.2f}m)")

# 시뮬레이션 실행
print("[연산] Ray Tracing 다시 시작...")
solver = PathSolver()
paths = solver(scene, max_depth=5, samples_per_src=1000000, diffuse_reflection=True, diffraction=True)

# 결과 확인
print("[시각화] 3D 뷰어 실행 (초록색/마커 확인)")
scene.preview(paths=paths, show_devices=True)

[보정] 빨간색 도로 좌표 정밀 복원 시작...
 -> Raw v_pos[412]: [ 4.1827621e+02  1.8871996e-14 -3.0820306e+02]
 -> 원본 데이터가 이미 올바른 좌표계입니다. 회전 없이 사용합니다.
[완료] 최종 경로 설정: 60개 점.
 -> 시작점(412): [ 418.2762     1.5     -308.20306]
[설정] 기지국 및 차량 재배치 완료.
[시각화 설정 오류] __setitem__(): incompatible function arguments. The following argument types are supported:
    1. __setitem__(self, arg0: str, arg1: bool, /) -> None
    2. __setitem__(self, arg0: str, arg1: int, /) -> None
    3. __setitem__(self, arg0: str, arg1: float, /) -> None
    4. __setitem__(self, arg0: str, arg1: str, /) -> None
    5. __setitem__(self, arg0: str, arg1: drjit.scalar.Array3f64, /) -> None
    6. __setitem__(self, arg0: str, arg1: mitsuba.ScalarColor3f, /) -> None
    7. __setitem__(self, arg0: str, arg1: mitsuba.ScalarColor3d, /) -> None
    8. __setitem__(self, arg0: str, arg1: mitsuba.ScalarAffineTransform3f, /) -> None
    9. __setitem__(self, arg0: str, arg1: mitsuba.ScalarAffineTransform3d, /) -> None
    10. __setitem__(self, arg0: s

In [25]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sionna.rt import Camera, PathSolver, PlanarArray, Transmitter, Receiver

# ==============================================================================
# 1. 단일 지점(412번) 좌표 설정
# ==============================================================================
# 사용자 요청 좌표 (412번)
target_pos_412 = np.array([418.28, 1.50, -308.20]) # 높이 1.5m 포함

print("="*60)
print(f"[설정] 테스트 목표: 자동차를 {target_pos_412}에 고정하고 전파 수신 확인")
print("="*60)

# ==============================================================================
# 2. 기지국(Tx) 및 자동차(Rx) 고정 배치
# ==============================================================================
# 기존 객체 제거
for name in ['Tx_1', 'Tx_2', 'Tx_3', 'Car_Marker']:
    if name in scene.transmitters: scene.remove(name)
if 'rx_car' in scene.receivers: scene.remove('rx_car')

# 안테나 설정
scene.tx_array = PlanarArray(num_rows=1, num_cols=4, pattern="tr38901", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

# 기지국 배치 (Tx_1만 활성화하여 확인하거나 전체 배치)
tx_positions = [
    [-125.663, 56.367, -181.453],
    [323.472, 36.869, -204.315], 
    [0.663, 56.367, -181.453]
]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at(target_pos_412) # [중요] 기지국이 자동차를 바라보게 회전 (지향성 안테나 효과 극대화)
    scene.add(tx)
    print(f" -> {tx_names[i]} 배치 완료 (Target을 바라봄).")

# 자동차(Rx) 배치 - [핵심] 단일 좌표 고정
rx = Receiver(name="rx_car", position=target_pos_412)
rx.receive_antenna = scene.rx_array
scene.add(rx)

print(" -> 자동차(Rx) 배치 완료 (고정 위치).")

# ==============================================================================
# 3. Ray Tracing 시뮬레이션 (단일 스냅샷)
# ==============================================================================
print("[연산] 경로 계산 시작 (Single Point)...")

solver = PathSolver()
# diffuse_reflection=True: 산란(거친 표면 반사) 활성화 -> 신호 도달 확률 높임
# diffraction=True: 회절(모서리 꺾임) 활성화 -> 장애물 뒤 도달 확률 높임
paths = solver(scene, max_depth=5, samples_per_src=1000000, diffuse_reflection=True, diffraction=True)

# ==============================================================================
# 4. 결과 검증 (전파가 도달했는가?)
# ==============================================================================
a, tau = paths.cir()

# 데이터가 있는지 확인 (경로 개수 체크)
# 형상 예: [Tx, Rx, Paths, 1] (Time 차원이 1이거나 없음)
has_signal = False
if tf.size(a) > 0:
    # 텐서의 마지막 차원(시간)이나 뒤에서 두번째(경로) 확인
    # 리스트일 경우 텐서 변환
    if isinstance(a, list): a = tf.convert_to_tensor(a)
    
    # 0이 아닌 값이 있는지 확인 (경로가 하나라도 연결되었는지)
    # 복잡한 인덱싱 대신 전체 합계나 최대값으로 확인
    total_power = tf.reduce_sum(tf.abs(a))
    if total_power > 0:
        has_signal = True
        print(f"\n✅ [성공] 전파가 자동차에 도달했습니다! (Total Amplitude Sum: {total_power:.2e})")
    else:
        print("\n❌ [실패] 전파가 도달하지 못했습니다. (Power = 0)")
        print("   -> 기지국과 자동차 사이에 장애물이 있거나 거리가 너무 멉니다.")
else:
    print("\n❌ [실패] 계산된 경로가 없습니다.")

# ==============================================================================
# 5. 시각화 (확인 사살)
# ==============================================================================
# 카메라를 자동차 근처로 이동
cam = Camera(position=target_pos_412 + np.array([0, 100, 100]), look_at=target_pos_412)

print("[시각화] 3D 뷰어 실행")
print("       -> 빨간 선(Ray)들이 자동차(Rx) 점으로 모이는지 확인하세요.")
scene.preview(paths=paths, show_devices=True)

[설정] 테스트 목표: 자동차를 [ 418.28    1.5  -308.2 ]에 고정하고 전파 수신 확인
 -> Tx_1 배치 완료 (Target을 바라봄).
 -> Tx_2 배치 완료 (Target을 바라봄).
 -> Tx_3 배치 완료 (Target을 바라봄).
 -> 자동차(Rx) 배치 완료 (고정 위치).
[연산] 경로 계산 시작 (Single Point)...

❌ [실패] 계산된 경로가 없습니다.
[시각화] 3D 뷰어 실행
       -> 빨간 선(Ray)들이 자동차(Rx) 점으로 모이는지 확인하세요.


I0000 00:00:1769754315.346314   40708 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 19583 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:81:00.0, compute capability: 8.6
I0000 00:00:1769754315.348658   40708 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22321 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:c1:00.0, compute capability: 8.6
